# Robotic Perception: From Sensors to Maps

## Table of Contents

0. [Perception as an Inverse Problem](#0)
1. [Proprioception — Motion Priors and Their Limits](#1)
2. [Camera Geometry and Calibration](#2)
3. [Feature Detection and Matching](#3)
4. [Two-View Geometry](#4)
5. [Many-View Geometry and Bundle Adjustment](#5)
6. [Depth Sensing — From Passive to Active](#6)
7. [LiDAR — Probabilistic Models and Map Building](#7)
8. [SLAM Systems and Sensor Fusion](#8)

### Notation

| Symbol | Meaning |
|--------|---------|
| $\mathbf{x}_t \in \mathrm{SE}(3)$ | robot pose (position + orientation) |
| $m$ | map — landmarks, occupancy grid, or implicit representation |
| $z_t^k$ | measurement from sensor $k$ at time $t$ |
| $h^k(\mathbf{x}, m)$ | forward (measurement) model for sensor $k$ |
| $p(z \mid \mathbf{x}, m)$ | sensor likelihood |
| $r = z - h(\mathbf{x}, m)$ | measurement residual |
| $K$ | camera intrinsic matrix |
| $T_{cw} = [R_{cw} \mid t_{cw}]$ | world-to-camera extrinsic transform |
| $\pi(\mathbf{X}_c; K)$ | pixel projection after perspective divide and intrinsics |
| $\pi_d(\mathbf{X}_c; K, d)$ | distorted pixel projection |
| $\Sigma$ | covariance matrix (generic) |
| $Q, R$ | process / measurement noise covariance |
| $l(m_i)$ | log-odds of occupancy for cell $i$ |
| $\xi \in \mathfrak{se}(3)$ | pose perturbation in Lie algebra |

### Conventions

- World points are written $\mathbf{X}_w$ and camera-frame points $\mathbf{X}_c = R_{cw}\,\mathbf{X}_w + t_{cw}$.
- We use image coordinates $(u, v)$ with $u$ to the right and $v$ downward.
- $\pi(\mathbf{X}_c; K)$ denotes **pixel projection**; intrinsics are absorbed into $\pi$ unless stated otherwise.
- $\pi_d(\mathbf{X}_c; K, d)$ denotes the same projection after distortion parameters $d$ are applied.
- Pose perturbations use a **left** update $T \leftarrow \exp(\hat\xi)\,T$ with twist ordering $\xi = (\rho, \phi)$, where $\rho \in \mathbb{R}^3$ is translation and $\phi \in \mathbb{R}^3$ is rotation.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt
import cv2

from lib.camera import (
    PinholeCamera, FisheyeCamera,
    project, project_jacobian,
    apply_distortion, undistort_points,
    calibrate_camera_from_checkerboard,
    calibrate_fisheye_from_frames,
    undistort_fisheye_image,
    run_fisheye_calibration_live,
    save_calibration, load_calibration,
)
from lib.encoder import EncoderModel, simulate_encoder
from lib.imu import ImuModel, simulate_imu_drift, ImuLiveDriftTracker
from lib.features import (
    harris_corner_response, detect_and_describe_orb,
    match_features_ransac,
)
from lib.aruco import detect_aruco, estimate_aruco_pose, detect_and_overlay_live
from lib.stereo import (
    rectify_stereo, compute_disparity,
    disparity_to_depth, triangulate_points,
)
from lib.sfm import two_view_sfm, bundle_adjustment_demo
from lib.depth import ToFModel, TSDFFusion, backproject_depth_frame
from lib.lidar import (
    BeamModel, LikelihoodField, OccupancyGrid,
    icp_point_to_point, icp_point_to_plane,
)
from lib.live import (
    RgbFrame, DepthFrame, ImuSample, ServoSample,
    SourceMode, SyncBuffer, LiveDemoSession,
)
from lib.servo import FeetechServoSource, SimulatedServoSource, make_servo_source
from lib.phone_stream import (
    PhoneReplaySource, PhoneUSBSource, PhoneWebSocketSource,
    load_imu_csv, load_depth_ply, load_camera_frames,
)
from lib.ws_server import PerceptionWSServer
from lib.viz import (
    plot_factor_graph_overview, plot_factor_graph_annotated,
    plot_encoder_quantization, plot_imu_drift,
    plot_projection_interactive, plot_distortion_grid,
    plot_calibration_result, plot_aruco_detection,
    plot_harris_steps, plot_scale_space_keypoints,
    plot_feature_matching, plot_epipolar_geometry_3d, plot_epipolar_lines,
    plot_stereo_pipeline, plot_depth_noise_curves,
    plot_sfm_reconstruction, plot_ba_sparsity, plot_ba_convergence,
    plot_depth_comparison, plot_tsdf_evolution,
    plot_beam_model_interactive, plot_occupancy_grid_evolution,
    plot_icp_convergence, plot_failure_gallery,
    LivePlot,
    create_live_encoder_plot, create_live_imu_drift_plot,
    create_live_camera_plot, create_live_aruco_plot, create_live_depth_plot,
)

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                      "grid.alpha": 0.2, "font.size": 10})

<a id="0"></a>
# 0. Perception as an Inverse Problem

A robot interacts with the world through **sensors**. Each sensor
converts a physical quantity into a signal corrupted by noise. The
central challenge of perception is to **invert** this process: given
noisy measurements, recover the robot's state and the structure of the
environment.

In the previous lecture we built the **inference engine** — Bayes
filters, Kalman filters, particle filters. Those algorithms all
require a **sensor model** $p(z \mid \mathbf{x}, m)$ as input. This
lecture derives those models for the sensors that matter most in
robotics.

---

## Unified sensor template

Every sensor block in this lecture follows the same structure:

$$
\underbrace{\text{physical quantity}}_{\text{pose, depth, reflectance}}
\;\xrightarrow{\text{transducer}}\;
\underbrace{\text{raw signal}}_{\text{voltage, counts, pixels}}
\;\xrightarrow{\text{noise}}\;
\underbrace{z = h(\mathbf{x}, m) + \varepsilon}_{\text{forward model}}
$$

From the forward model we get:
- **Residual** $r = z - h(\mathbf{x}, m)$ — the building block of
  least-squares optimisation (factor graphs, bundle adjustment, SLAM).
- **Inverse model** $p(m_i \mid z, \mathbf{x})$ — used to update
  occupancy grids and other map representations.

---

## Factor graph preview

A **factor graph** connects unknown variables (robot poses, map) through
factors, each contributed by a single measurement or prior. Every sensor
we study today adds a specific type of factor:

| Factor type | Source | Connects |
|-------------|--------|----------|
| motion / odometry | encoders, IMU | consecutive poses |
| reprojection | camera features | pose ↔ landmark |
| depth | stereo, ToF, LiDAR | pose ↔ map point |
| scan-matching | LiDAR ICP/NDT | consecutive poses |
| absolute | GPS, beacons | single pose |

The figure below will be annotated section by section as we derive each
factor.

In [ ]:
plot_factor_graph_overview();

<a id="1"></a>
# 1. Proprioception — Motion Priors and Their Limits

Proprioceptive sensors measure the robot's **own state** — joint
angles, angular rates, accelerations. They provide the **motion model**
$p(\mathbf{x}_t \mid \mathbf{x}_{t-1}, u_t)$ that chains poses
together. But they all drift.

---

## 1.1 Motor Encoders

**Physics.** An optical or magnetic encoder produces $N$ discrete
ticks per revolution. The measured angle is quantised:

$$
\hat\theta = \mathrm{round}\!\left(\frac{\theta}{\Delta}\right)\Delta,
\qquad \Delta = \frac{2\pi}{N}
$$

The quantisation error $e = \hat\theta - \theta$ is approximately
uniform on $[-\Delta/2,\; \Delta/2]$, giving variance

$$
\sigma_q^2 = \frac{\Delta^2}{12}
\tag{1.1}
$$

**Velocity estimation.** Differentiating discrete ticks amplifies
noise: $\hat\omega = \Delta\hat\theta / \Delta t$ has variance
$\sigma_q^2 / \Delta t^2$, which blows up at high sampling rates.

**Covariance propagation.** For a differential-drive robot with
wheel displacements $\Delta s_L, \Delta s_R$ and baseline $b$, the
pose increment $(x, y, \theta)$ depends on
$\Delta s = (\Delta s_R + \Delta s_L)/2$ and
$\Delta\theta = (\Delta s_R - \Delta s_L)/b$.
Linearising the pose update gives

$$
\Sigma_{\mathbf{x}} = J\,\Sigma_{\Delta s}\,J^\top,
\qquad
\Sigma_{\Delta s} =
\begin{pmatrix} \sigma_L^2 & 0 \\ 0 & \sigma_R^2 \end{pmatrix},
\quad \sigma_{L,R}^2 \propto |\Delta s_{L,R}|
\tag{1.2}
$$

so pose uncertainty **grows with distance traveled**.

**Observability.** Encoders give **relative** motion only — no
absolute position, no orientation without a compass.

**Failure modes.**
- *Wheel slip* — heavy-tailed, non-Gaussian errors (mud, ice, fast
  turns). A mixture model
  $p(\Delta s) = (1-\epsilon)\,\mathcal{N} + \epsilon\,p_{\text{slip}}$
  is more realistic.
- *Backlash* — dead zone where input motion produces no encoder
  change.
- *Calibration error* — wrong wheel radius or baseline propagates
  systematically.

In [ ]:
encoder = EncoderModel(ticks_per_rev=100)
true_angle, quantized_angle, t = simulate_encoder(encoder, duration=2.0, omega=3.0, dt=0.001)
plot_encoder_quantization(t, true_angle, quantized_angle, encoder.delta);

In [ ]:
# ── Live Demo: Feetech Servo Encoder ────────────────────────────────────────
# Tries to connect to a real Feetech STS3215 servo over serial.
# Falls back to a simulated encoder if no hardware is found.
#
# Stop the loop with the ■ (interrupt kernel) button.
# ─────────────────────────────────────────────────────────────────────────────
import time
from lib.servo import make_servo_source
from lib.viz import create_live_encoder_plot

_servo = make_servo_source(port="/dev/tty.usbmodem5AB01587771", servo_id=1, poll_hz=100.0)
_servo.start()
print(f"Source mode: {_servo.mode.value}")

_enc_plot = create_live_encoder_plot()
_enc_plot.show()

try:
    for _ in range(600):          # ~12 s at 50 ms update interval
        samples = _servo.get_recent(n=200)
        if samples:
            _enc_plot.update(samples)
        time.sleep(0.05)
finally:
    _servo.stop()
    _enc_plot.close()
    print("Servo demo stopped.")

## 1.2 Inertial Measurement Unit (IMU)

**Physics.** A MEMS IMU contains a 3-axis gyroscope and a 3-axis
accelerometer. We measure angular velocity in the **body frame** and
specific force in the **body frame**. With the convention
$R_{bw} = R_{wb}^\top$, the measurement model is

$$
\omega_m = \omega_b + b_\omega + n_\omega, \qquad
a_m = R_{bw}(a_w - g_w) + b_a + n_a
\tag{1.3}
$$

where $a_w$ and $g_w$ live in the world frame, while
$\omega_m, a_m, b_\omega, b_a$ live in the body frame. The biases evolve
as random walks:

$$
b_{t+1} = b_t + w_b, \qquad w_b \sim \mathcal{N}(0, Q_b)
\tag{1.4}
$$

**Drift accumulation.** Integrating gyro noise gives **orientation
drift** $\propto \sqrt{t}$. Double-integrating accelerometer noise gives
**position drift** $\propto t^{3/2}$ — unbounded and fast. A
stationary IMU's estimated position wanders metres in seconds.

**Observability.**
- Gyroscope: relative orientation change.
- Accelerometer at rest: gravity direction, hence roll and pitch up to
  accelerometer bias. **Not** yaw, and **not** position without
  external reference.
- Scale is not observable from IMU alone in visual-inertial systems
  until sufficient rotational or translational excitation.

**Failure modes.**
- Unbounded position drift from double integration.
- Bias instability (temperature-dependent).
- Vibration aliasing at high frequencies.
- Saturation during violent manoeuvres.

In [ ]:
imu = ImuModel(gyro_noise=0.01, accel_noise=0.05, gyro_bias_drift=1e-4, accel_bias_drift=1e-3)
imu_data = simulate_imu_drift(imu, duration=60.0, dt=0.01, rng=rng)
plot_imu_drift(imu_data)

In [ ]:
# ── Live Demo: iPhone IMU Drift ──────────────────────────────────────────────
# Connects to the PerceptionDemo iOS app over USB via iproxy.
# If no iPhone connects within 10 s, replays pre-recorded data from
# assets/phone_recordings/ (or falls back to pure simulation).
# Before running: start `iproxy 7777 7777` on the Mac and tap
# Start Listening in the iPhone app.
#
# Stop the loop with the ■ (interrupt kernel) button.
# ─────────────────────────────────────────────────────────────────────────────
import time
from pathlib import Path
from lib.phone_stream import PhoneUSBSource, PhoneReplaySource
from lib.imu import ImuLiveDriftTracker
from lib.viz import create_live_imu_drift_plot

_phone_imu = PhoneUSBSource()
_phone_imu.start()
print("Waiting up to 10 s for iPhone USB connection…")
connected = _phone_imu.wait_for_connection(timeout=10.0)

if not connected:
    _phone_imu.stop()
    replay_dir = Path("assets/phone_recordings")
    if any(replay_dir.glob("imu*.csv")) or (replay_dir / "trajectory.jsonl").exists():
        print("No iPhone — replaying pre-recorded data.")
        _phone_imu = PhoneReplaySource(scan_dir=replay_dir, replay_fps=50.0)
    else:
        print("No iPhone and no replay data — simulating IMU drift instead.")
        _phone_imu = None

_tracker = ImuLiveDriftTracker(dt=0.01)
_imu_plot = create_live_imu_drift_plot()
_imu_plot.show()

try:
    if _phone_imu is not None:
        if hasattr(_phone_imu, 'start') and hasattr(_phone_imu, 'is_running') and not _phone_imu.is_running():
            _phone_imu.start()
        for _ in range(2000):   # ~20 s at 10 ms polling
            sample = _phone_imu.sync_buffer.get_latest_imu() if hasattr(_phone_imu, 'sync_buffer') \
                     else _phone_imu.get_latest_imu()
            if sample:
                _tracker.update(sample)
            if len(_tracker.timestamps) % 50 == 1:
                _imu_plot.update(_tracker)
            time.sleep(0.01)
finally:
    if _phone_imu is not None and hasattr(_phone_imu, 'stop'):
        _phone_imu.stop()
    _imu_plot.close()
    print(f"IMU demo stopped. Collected {len(_tracker.timestamps)} samples.")

### What can we observe so far?

Encoders and IMU contribute **motion factors** — edges between
consecutive poses in the factor graph. They constrain **relative**
motion, and a static accelerometer gives the gravity direction
(roll/pitch), but they still accumulate drift without bound and do not
provide global position or yaw.

> **Key insight:** motion sensors alone cannot build a consistent map
> or maintain a global position estimate. Everything that follows adds
> **measurement factors** that anchor the trajectory to the world.

<a id="2"></a>
# 2. Camera Geometry and Calibration

A camera is the richest single sensor available to a robot — every
pixel carries information about the 3D world. But extracting that
information requires precise geometric models.

---

## 2.1 Pinhole Camera Model

**Derivation from similar triangles.** A point
$\mathbf{X}_c = (X_c, Y_c, Z_c)^\top$ in camera coordinates projects
onto the image plane at distance $f$ (focal length) by similar
triangles:

$$
u' = f\,\frac{X_c}{Z_c}, \qquad v' = f\,\frac{Y_c}{Z_c}
$$

In pixel coordinates, accounting for different focal lengths per axis
$(f_x, f_y)$, principal point $(c_x, c_y)$, and (rarely) skew $s$:

$$
K = \begin{pmatrix} f_x & s & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{pmatrix}
\tag{2.1}
$$

**Homogeneous coordinates.** Introducing
$\tilde{\mathbf{u}} = (u, v, 1)^\top$ and
$\tilde{\mathbf{X}}_w = (X_w, Y_w, Z_w, 1)^\top$, the full projection
becomes a matrix multiplication followed by normalisation:

$$
\lambda\,\tilde{\mathbf{u}} = K\,T_{cw}\,\tilde{\mathbf{X}}_w
= K\,[R_{cw} \mid t_{cw}]\,\tilde{\mathbf{X}}_w,
\qquad
\mathbf{u} = \pi(\mathbf{X}_c; K)
\tag{2.2}
$$

where $T_{cw}$ maps world coordinates to camera coordinates and
$\mathbf{X}_c = R_{cw}\,\mathbf{X}_w + t_{cw}$.

**Projection Jacobian.** The pixel projection
$\pi(\mathbf{X}_c; K) = \bigl(f_x X_c/Z_c + c_x,\; f_y Y_c/Z_c + c_y\bigr)$
has Jacobian

$$
\frac{\partial \pi}{\partial \mathbf{X}_c}
= \begin{pmatrix}
f_x / Z_c & 0 & -f_x X_c / Z_c^2 \\
0 & f_y / Z_c & -f_y Y_c / Z_c^2
\end{pmatrix}
\tag{2.3}
$$

where we neglect the skew term in the Jacobian because modern camera
calibrations typically have $s \approx 0$. This $2 \times 3$ matrix
reappears in every vision-based optimisation (bundle adjustment, visual
SLAM, PnP refinement).

In [ ]:
plot_projection_interactive();

## 2.2 Lens Distortion

Real lenses deviate from the pinhole ideal. The **Brown–Conrady** model
corrects normalised coordinates $(x_n, y_n)$ as follows.

**Radial distortion** (barrel / pincushion / mustache):

$$
x_r = x_n\,(1 + k_1 r^2 + k_2 r^4 + k_3 r^6), \qquad
y_r = y_n\,(1 + k_1 r^2 + k_2 r^4 + k_3 r^6), \qquad
r^2 = x_n^2 + y_n^2
\tag{2.4}
$$

**Tangential distortion** (decentring):

$$
\delta x = 2\,p_1\,x_n y_n + p_2(r^2 + 2x_n^2), \qquad
\delta y = p_1(r^2 + 2y_n^2) + 2\,p_2\,x_n y_n
\tag{2.5}
$$

The final distorted point is

$$
x_d = x_r + \delta x, \qquad y_d = y_r + \delta y
\tag{2.6}
$$

**Fisheye models.** For lenses with FOV $\gtrsim 120°$, the
Brown–Conrady polynomial diverges. Fisheye models map the incidence
angle $\theta$ to image radius directly:

- *Equidistant*: $r = f\,\theta$
- *Equisolid-angle*: $r = 2f\sin(\theta/2)$
- *Kannala–Brandt* (OpenCV `cv2.fisheye`):
  $\theta_d = \theta(1 + k_1\theta^2 + k_2\theta^4 + k_3\theta^6 + k_4\theta^8)$

The grid below visualises Brown–Conrady radial and tangential cases; a
true fisheye image is better understood from real camera data.

**Failure modes.**
- Applying a pinhole model to a fisheye image produces unusable
  results.
- Extrapolating distortion parameters outside the calibration range
  creates artefacts at image borders.

In [ ]:
plot_distortion_grid();

In [ ]:
# ── Live Demo: Fisheye Camera Calibration + Undistortion ────────────────────
# Connects to the USB fisheye camera (device index 0) and runs rolling
# checkerboard calibration. Point the camera at a 9×6 printed checkerboard.
# Results are cached in assets/fisheye_sample/calibration.json.
# Falls back to pre-captured images if no camera is available.
#
# Stop the calibration loop with the ■ (interrupt kernel) button.
# ─────────────────────────────────────────────────────────────────────────────
import cv2
import time
from pathlib import Path
from lib.camera import (
    run_fisheye_calibration_live, undistort_fisheye_image,
    save_calibration, load_calibration, FisheyeCamera,
)
from lib.viz import create_live_camera_plot

CALIB_PATH = Path("assets/fisheye_sample/calibration.json")
CALIB_PATH.parent.mkdir(parents=True, exist_ok=True)

fisheye_cam = None

if CALIB_PATH.exists():
    fisheye_cam = load_calibration(CALIB_PATH)
    print(f"Loaded fisheye calibration from {CALIB_PATH}")
    print(f"  K =\n{fisheye_cam.K}")
    print(f"  D = {fisheye_cam.D.ravel()}")
else:
    print("No saved calibration found. Starting live calibration…")
    print("Point the fisheye camera at a 9×6 checkerboard.")
    print("Stop with ■ when reprojection error stabilises.")

    _cam_plot = create_live_camera_plot()
    _cam_plot.show()

    try:
        fisheye_cam = run_fisheye_calibration_live(
            source=1,
            display_fn=lambda r, u: _cam_plot.update((r, u)),
        )
        save_calibration(CALIB_PATH, fisheye_cam)
        print(f"Calibration saved to {CALIB_PATH}")
        print(f"  K =\n{fisheye_cam.K}")
        print(f"  D = {fisheye_cam.D.ravel()}")
    except RuntimeError as e:
        print(f"Calibration failed: {e}")
        fisheye_samples = sorted(Path("assets/fisheye_sample").glob("*.png"))
        if fisheye_samples:
            print("Showing pre-captured sample instead.")
    finally:
        _cam_plot.close()

# Show one undistorted frame if we have a calibration
if fisheye_cam is not None:
    try:
        cap = cv2.VideoCapture(1)
        ok, frame = cap.read()
        cap.release()
        if ok:
            undist = undistort_fisheye_image(frame, fisheye_cam)
            import matplotlib.pyplot as plt
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            axes[0].set_title("Raw fisheye"); axes[0].axis("off")
            axes[1].imshow(cv2.cvtColor(undist, cv2.COLOR_BGR2RGB))
            axes[1].set_title("Undistorted"); axes[1].axis("off")
            plt.tight_layout(); plt.show()
    except Exception as e:
        print(f"Could not capture live frame: {e}")

## 2.3 Camera Calibration

**Zhang's method** (2000) recovers $K$ and distortion coefficients from
multiple views of a planar pattern (checkerboard).

**Key idea.** Each view of a planar target defines a **homography**
$H_i$ relating world points on the plane to image points:
$\lambda\,\tilde{\mathbf{u}} = H_i\,\tilde{\mathbf{X}}_{\text{plane}}$.
Since $H_i = K\,[r_1 \mid r_2 \mid t]_i$ and $R$ is a rotation, the
columns of $H_i$ impose constraints on $K$. Two constraints per view →
three or more views suffice to recover $K$ in closed form, then refine
by minimising **reprojection error**:

$$
\min_{K,\,d,\,\{R_{cw,i}, t_{cw,i}\}} \sum_{i,j}
\left\| \mathbf{u}_{ij} - \pi_d\!\left(R_{cw,i}\,\mathbf{X}_j + t_{cw,i}; K, d\right) \right\|^2
\tag{2.7}
$$

where $\pi_d$ includes distortion and $\mathbf{u}_{ij}$ are detected
corners. Reprojection error is the gold-standard quality metric because
it is the same residual later minimized in bundle adjustment; a good
calibration usually achieves $< 0.5\,\text{px}$ RMS.

### Calibration hierarchy

In a full robotic system, calibration is layered:

1. **Intrinsics** ($K$) and **distortion** ($k_1, k_2, \ldots, p_1, p_2$) — per camera.
2. **Stereo extrinsics** — relative pose between left and right cameras.
3. **Camera-body extrinsics** — transform from camera frame to robot body frame.
4. **Camera-IMU extrinsics + temporal offset** — for visual-inertial systems.

Each level adds parameters and has its own failure modes (e.g., thermal
expansion changing stereo baseline, imprecise time synchronisation
introducing motion blur in VIO).

In [ ]:
calib_images = sorted(Path("assets/checkerboard").glob("*.png"))
if calib_images:
    K_est, dist_est, reproj_err, rvecs, tvecs, calib_data = calibrate_camera_from_checkerboard(
        calib_images, board_size=(9, 6), square_size=0.025
    )
    print(f"Estimated K:\n{K_est}")
    print(f"\nDistortion: {dist_est.ravel()}")
    print(f"Mean reprojection error: {reproj_err:.3f} px")
    plot_calibration_result(calib_images, K_est, dist_est, rvecs, tvecs, reproj_err, calib_data)
else:
    print("No calibration images found in assets/checkerboard/ — "
          "run the live demo or add pre-captured images.")

## 2.4 Pose from Known Geometry: ArUco Markers and PnP

ArUco markers are the simplest "pixels → pose" pipeline: the 3D
geometry is **known** (a square of known side length), so a single
detected marker plus a calibrated camera yield a **full 6-DOF pose**.

**Detection pipeline.**
1. Adaptive threshold → candidate quadrilaterals.
2. Decode binary pattern → marker ID from dictionary.
3. Refine corner sub-pixel locations.

**PnP (Perspective-n-Point).** Given $n \geq 3$ correspondences
between known 3D points $\{\mathbf{X}_j\}$ and their observed
projections $\{\mathbf{u}_j\}$, plus intrinsics $K$, solve for
camera pose $(R_{cw}, t_{cw})$:

$$
\min_{R_{cw}, t_{cw}} \sum_{j=1}^{n}
\left\| \mathbf{u}_j - \pi\!\left(R_{cw}\,\mathbf{X}_j + t_{cw}; K\right) \right\|^2
\tag{2.8}
$$

P3P gives up to 4 algebraic solutions from 3 points; a 4th point
disambiguates. `cv2.solvePnP` uses EPnP + Levenberg–Marquardt
refinement.

**ChArUco boards** combine a checkerboard (accurate sub-pixel corners)
with ArUco markers (robust detection even under partial occlusion).
They are ideal for simultaneous calibration and pose estimation.

**Observability.** A single marker provides full 6-DOF pose **if** the
camera is calibrated and the marker size is known. Multiple markers
overdetermine the system and reduce noise.

**Failure modes.**
- Pose ambiguity from coplanar points at grazing angles.
- Marker too distant — pixel noise dominates corner localisation.
- Partial occlusion — need ≥ 4 visible corners for PnP.

In [ ]:
aruco_images = sorted(Path("assets/aruco_board").glob("*.png"))
if aruco_images:
    img = cv2.imread(str(aruco_images[0]))
    corners, ids, poses = detect_aruco(
        img,
        K_est if 'K_est' in dir() else None,
        dist_est if 'dist_est' in dir() else None,
    )
    plot_aruco_detection(
        img,
        corners,
        ids,
        poses,
        K_est if 'K_est' in dir() else None,
        dist_est if 'dist_est' in dir() else None,
    )
else:
    print("No ArUco images found — add pre-captured images to assets/aruco_board/.")

### What can we observe so far?

- A calibrated camera gives **bearing** to features (direction, not
  depth).
- With **known geometry** (ArUco markers), we get full 6-DOF pose
  — the first absolute anchor in the factor graph.
- Camera reprojection factors connect pose nodes to landmark nodes.

> **Next question:** what if landmarks are *not* known in advance?
> We need to **discover** them — that requires feature detection.

In [ ]:
# ── Live Demo: ArUco Detection from iPhone Camera ────────────────────────────
# Uses the iPhone RGB stream to show live ArUco detection + pose axes.
# Requires PerceptionDemo app over USB (`iproxy 7777 7777`, then tap Start Listening).
# Falls back to pre-captured images from assets/aruco_board/.
#
# Stop with the ■ (interrupt kernel) button.
# ─────────────────────────────────────────────────────────────────────────────
import time
from pathlib import Path
import numpy as np
import cv2
from lib.phone_stream import PhoneUSBSource
from lib.aruco import detect_and_overlay_live
from lib.viz import create_live_aruco_plot

# Connect to the iPhone app over USB/iproxy
try:
    _aruco_phone = PhoneUSBSource()
    _aruco_phone.start()
    print("Waiting up to 10 s for iPhone USB connection…")
    connected = _aruco_phone.wait_for_connection(timeout=10.0)
except Exception:
    connected = False

_aruco_plot = create_live_aruco_plot()
_aruco_plot.show()

if connected:
    # Use iPhone K from the first received frame, fall back to a rough estimate
    _K_aruco = None
    print("iPhone connected — starting ArUco detection.")
    try:
        for _ in range(300):  # ~30 s at 100 ms polling
            frame_pkt = _aruco_phone.get_latest_rgb()
            if frame_pkt is not None:
                frame = frame_pkt.image.copy()
                K = frame_pkt.intrinsics if not np.allclose(frame_pkt.intrinsics, np.eye(3)) \
                    else np.array([[frame.shape[1]*0.9, 0, frame.shape[1]/2],
                                   [0, frame.shape[1]*0.9, frame.shape[0]/2],
                                   [0, 0, 1]], dtype=np.float64)
                annotated, _ = detect_and_overlay_live(frame, K)
                _aruco_plot.update(annotated)
            time.sleep(0.1)
    finally:
        _aruco_phone.stop()
        _aruco_plot.close()
        print("ArUco live demo stopped.")
else:
    print("No iPhone — using pre-captured images from assets/aruco_board/")
    if hasattr(_aruco_phone, 'stop'):
        _aruco_phone.stop()
    aruco_imgs = sorted(Path("assets/aruco_board").glob("*.png"))
    if aruco_imgs:
        for p in aruco_imgs[:3]:
            img = cv2.imread(str(p))
            if img is not None:
                K_fallback = np.array([[img.shape[1]*0.9, 0, img.shape[1]/2],
                                       [0, img.shape[1]*0.9, img.shape[0]/2],
                                       [0, 0, 1]], dtype=np.float64)
                annotated, _ = detect_and_overlay_live(img.copy(), K_fallback)
                _aruco_plot.update(annotated)
                time.sleep(0.5)
    _aruco_plot.close()

<a id="3"></a>
# 3. Feature Detection and Matching

For SLAM and structure from motion we need to find and track **the same
physical point** across images taken from different viewpoints. This
requires: (1) detecting distinctive image locations, (2) describing
them, and (3) matching descriptions across views.

---

## 3.1 Harris Corner Detector

**Intuition.** Shift a small window across the image. In a flat region
the image barely changes; along an edge it changes in one direction
only; at a corner it changes in **all** directions.

**Start from patch SSD.** For a small displacement
$\Delta = (\Delta x, \Delta y)^\top$, measure the intensity change in a
window $w$ by

$$
E(\Delta) = \sum_{(x,y) \in w} w(x,y)\,[I(x+\Delta x, y+\Delta y) - I(x,y)]^2
$$

Using a first-order Taylor approximation,
$I(x+\Delta x, y+\Delta y) \approx I + I_x\Delta x + I_y\Delta y$, so

$$
E(\Delta) \approx
\begin{pmatrix} \Delta x & \Delta y \end{pmatrix}
\left(\sum_{(x,y) \in w} w(x,y)
\begin{pmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{pmatrix}\right)
\begin{pmatrix} \Delta x \\ \Delta y \end{pmatrix}
\tag{3.1}
$$

The middle matrix is the **second moment matrix** (structure tensor):

$$
M = \sum_{(x,y) \in w} w(x,y)
\begin{pmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{pmatrix}
\tag{3.2}
$$

$M$ is a $2 \times 2$ symmetric positive-semidefinite matrix with
eigenvalues $\lambda_1 \geq \lambda_2 \geq 0$.

**Classification by eigenvalues:**

| Region | $\lambda_1$ | $\lambda_2$ |
|--------|-------------|-------------|
| Flat   | $\approx 0$ | $\approx 0$ |
| Edge   | large | $\approx 0$ |
| Corner | large | large |

**Cornerness function.** Computing eigenvalues for every pixel is
expensive. Harris uses:

$$
R = \det(M) - k\,\mathrm{tr}(M)^2 = \lambda_1\lambda_2 - k(\lambda_1 + \lambda_2)^2
\tag{3.3}
$$

with $k \approx 0.04$–$0.06$. A pixel is a corner if $R > \tau$.
Shi–Tomasi uses $\min(\lambda_1, \lambda_2) > \tau$ instead. In
practice we also apply **non-maximum suppression** so the detector
returns isolated interest points rather than whole high-response
regions.

In [ ]:
scene_images = sorted(Path("assets/scene_images").glob("*.png"))
if scene_images:
    img_gray = cv2.imread(str(scene_images[0]), cv2.IMREAD_GRAYSCALE)
else:
    img_gray = np.random.default_rng(0).integers(0, 256, (480, 640), dtype=np.uint8)
    cv2.rectangle(img_gray, (100, 100), (300, 300), 200, -1)
    cv2.rectangle(img_gray, (350, 150), (550, 350), 80, -1)
    cv2.circle(img_gray, (450, 250), 60, 160, -1)

response, eigenvalues = harris_corner_response(img_gray, k=0.04, block_size=5)
plot_harris_steps(img_gray, response, eigenvalues)

## 3.2 Scale-Invariant Features (SIFT / ORB)

Harris corners are not **scale-invariant**: a corner at one resolution
may be an edge at another. Scale-invariant detectors search across a
**scale space** — a stack of progressively blurred images.

**SIFT** (Lowe, 2004): Difference-of-Gaussians (DoG) pyramid →
extrema across scale → orientation from gradient histogram → 128-D
descriptor (histograms of oriented gradients in $4\times4$ cells).

**ORB** (Rublee et al., 2011): FAST corner detector on a pyramid →
orientation via intensity centroid → BRIEF binary descriptor (256-bit),
rotated to canonical orientation. ~100× faster than SIFT.

| | SIFT | ORB |
|--|------|-----|
| Descriptor | 128-D float | 256-bit binary |
| Invariance | scale + rotation | scale + rotation |
| Speed | slow | real-time |
| Matching | L2 distance | Hamming distance |

**Failure modes.**
- *Repetitive texture* — many similar descriptors, ambiguous matches.
- *Motion blur* — gradients smeared, keypoints unstable.
- *Extreme viewpoint change* — descriptors not affine-invariant.

---

## 3.3 Feature Matching and Outlier Rejection

**Matching.** For each descriptor in image 1, find the closest
descriptor in image 2 (brute-force or approximate via FLANN).

**Lowe's ratio test.** Reject a match if the distance to the best
match is close to the second-best:
$d_1 / d_2 > 0.75 \Rightarrow$ reject. This eliminates ambiguous
matches in repetitive scenes.

**RANSAC.** Even after ratio test, outliers remain. RANSAC:
1. Sample minimal set (e.g., 8 points for $F$, 5 for $E$).
2. Fit model (fundamental or essential matrix).
3. Count inliers ($|\mathbf{x}'^{\!\top} F\,\mathbf{x}| < \epsilon$).
4. Repeat; keep model with most inliers.

RANSAC connects to robust estimation in SLAM — the same idea of
separating inliers from outliers underpins robust kernels (Huber,
Cauchy) used in factor-graph optimisation.

In [ ]:
if len(scene_images) >= 2:
    img1 = cv2.imread(str(scene_images[0]), cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(str(scene_images[1]), cv2.IMREAD_GRAYSCALE)
else:
    img1, img2 = img_gray, np.roll(img_gray, 40, axis=1)

kp1, des1 = detect_and_describe_orb(img1)
kp2, des2 = detect_and_describe_orb(img2)
plot_scale_space_keypoints(img1, kp1)

matches, inlier_mask = match_features_ransac(kp1, des1, kp2, des2)
plot_feature_matching(img1, kp1, img2, kp2, matches, inlier_mask)

<a id="4"></a>
# 4. Two-View Geometry

Given two images of the same scene, what can we recover about the 3D
structure and the relative camera motion?

---

## 4.1 Epipolar Geometry

Consider two cameras with centres $\mathbf{C}, \mathbf{C}'$ observing
a 3D point $\mathbf{X}$. The three points
$\mathbf{C}, \mathbf{C}', \mathbf{X}$ define the **epipolar plane**.
The intersection of this plane with each image gives the **epipolar
lines** — the locus of possible correspondences.

**Coplanarity constraint.** In calibrated normalised coordinates
$\hat{\mathbf{x}} = K^{-1}\tilde{\mathbf{u}}$, let the second camera see
points related by
$\mathbf{X}' = R\,\mathbf{X} + t$. Because
$\hat{\mathbf{x}}', t,$ and $R\hat{\mathbf{x}}$ lie in the same
plane, their scalar triple product is zero:

$$
\hat{\mathbf{x}}'^{\!\top} \bigl(t \times (R\hat{\mathbf{x}})\bigr) = 0
$$

Using $[t]_\times y = t \times y$, we obtain

$$
\hat{\mathbf{x}}'^{\!\top}\,[t]_\times R\,\hat{\mathbf{x}} = 0
\qquad \Longrightarrow \qquad
\hat{\mathbf{x}}'^{\!\top}\,E\,\hat{\mathbf{x}} = 0,
\qquad E = [t]_\times R
\tag{4.1}
$$

where $E$ is the **essential matrix** (5 DOF: 3 rotation + 2
translation direction; scale is lost).

**Fundamental matrix.** In pixel coordinates (uncalibrated):

$$
\mathbf{x}'^{\!\top}\,F\,\mathbf{x} = 0,
\qquad
F = K'^{-\top}\,E\,K^{-1}
\tag{4.2}
$$

$F$ has 7 DOF and rank 2 (it has a non-trivial null space: the
epipoles).

**8-point algorithm.** Each correspondence gives one linear equation
in the 9 entries of $F$. With $\geq 8$ correspondences, solve the
homogeneous system via SVD. The **normalised** variant (Hartley, 1997)
centres and scales coordinates first for numerical stability.

**Recovering $(R, t)$ from $E$.** SVD of $E$ gives 4 candidate
$(R, t)$ pairs. The correct one is selected by the **cheirality
check**: triangulated points must have positive depth in both cameras.

**Observability.** From two views we recover:
- **Rotation** $R$ — fully.
- **Translation direction** $t / \|t\|$ — but **not** the scale $\|t\|$.
- This is the fundamental **scale ambiguity** of monocular
  reconstruction.

In [ ]:
plot_epipolar_geometry_3d();

In [ ]:
stereo_left = sorted(Path("assets/stereo_pair").glob("*left*"))
stereo_right = sorted(Path("assets/stereo_pair").glob("*right*"))
if stereo_left and stereo_right:
    imgL = cv2.imread(str(stereo_left[0]), cv2.IMREAD_GRAYSCALE)
    imgR = cv2.imread(str(stereo_right[0]), cv2.IMREAD_GRAYSCALE)
else:
    imgL, imgR = img1, img2

kpL, desL = detect_and_describe_orb(imgL)
kpR, desR = detect_and_describe_orb(imgR)
matches_epi, mask_epi = match_features_ransac(kpL, desL, kpR, desR)
plot_epipolar_lines(imgL, imgR, kpL, kpR, matches_epi, mask_epi)

## 4.2 Triangulation

Given two camera matrices $P, P'$ and a correspondence
$\mathbf{u}, \mathbf{u}'$, find $\mathbf{X}$ such that
$\lambda\,\tilde{\mathbf{u}} = P\,\tilde{\mathbf{X}}$ and
$\lambda'\tilde{\mathbf{u}}' = P'\tilde{\mathbf{X}}$.

**DLT.** Eliminate $\lambda$ by cross product:
$\mathbf{u} \times (P\,\tilde{\mathbf{X}}) = 0$ gives two independent
equations per view → 4 equations, solve for $\tilde{\mathbf{X}}$ via
SVD. Geometrically: find the point closest to two skew rays.

**Uncertainty.** Triangulation accuracy degrades at **low parallax**
(small baseline-to-depth ratio). In rectified stereo,
$Z = fB/d$, so

$$
\sigma_Z = \left|\frac{\partial Z}{\partial d}\right|\sigma_d
= \frac{Z^2}{fB}\,\sigma_d
$$

When $B/Z \to 0$, both rays become nearly parallel and the depth
estimate becomes ill-conditioned.

**Failure modes.**
- Near-zero baseline → degenerate (pure rotation).
- Forward motion → all epipolar lines converge to the epipole; depth
  accuracy collapses for points near the focus of expansion.

---

## 4.3 Stereo Vision as Constrained Two-View

A calibrated stereo rig is two-view geometry with **known** baseline
$B$ and parallel optical axes after **rectification**.

**Rectification** warps both images so epipolar lines become horizontal
scan lines. Correspondences then lie on the same row, reducing the
search to 1D. In practice, one first estimates the true left-right
extrinsics and only then rectifies. The notebook demo below uses a
simplified horizontal-baseline teaching model to isolate the geometry.

**Disparity and depth.** For a point at depth $Z$:

$$
d = u_L - u_R = \frac{f\,B}{Z}
\qquad \Longrightarrow \qquad
Z = \frac{f\,B}{d}
\tag{4.3}
$$

**Noise propagation.** If disparity noise is $\sigma_d$ (typically
$\sim 0.5$–$1$ pixel):

$$
\sigma_Z = \left|\frac{\partial Z}{\partial d}\right| \sigma_d
= \frac{f\,B}{d^2}\,\sigma_d
= \frac{Z^2}{f\,B}\,\sigma_d
\tag{4.4}
$$

Depth uncertainty grows **quadratically** with distance — the
fundamental limitation of passive stereo.

In [ ]:
K_stereo = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1]], dtype=np.float64)
baseline = 0.12

if stereo_left and stereo_right:
    imgL_rect, imgR_rect = rectify_stereo(imgL, imgR, K_stereo, baseline)
    disparity = compute_disparity(imgL_rect, imgR_rect)
    depth = disparity_to_depth(disparity, K_stereo[0, 0], baseline)
    plot_stereo_pipeline(imgL, imgR, imgL_rect, imgR_rect, disparity, depth)
else:
    print("No stereo pair found — add images to assets/stereo_pair/.")

plot_depth_noise_curves(f=500, baselines=[0.06, 0.12, 0.25], sigma_d=0.5)

### What can we observe so far?

- Two views give 3D structure **up to scale** (monocular).
- A calibrated stereo rig gives **metric depth** (known baseline),
  but with quadratically growing uncertainty.
- In the factor graph: landmark triangulation + camera factors produce
  3D reconstruction. With two views there is **gauge freedom** (7 DOF
  for monocular). More views and bundle adjustment tighten it.

<a id="5"></a>
# 5. Many-View Geometry and Bundle Adjustment

---

## 5.1 Structure from Motion Pipeline

**Incremental SfM** builds a 3D reconstruction by adding one view at a
time:

1. **Initialise** from a two-view pair: essential matrix → $(R, t)$ →
   triangulate.
2. **For each new view:**
   - Find 2D–3D correspondences (match features to already-triangulated
     points).
   - Solve PnP for the new camera pose.
   - Triangulate newly matched points.
3. **Refine** the entire reconstruction with bundle adjustment.

The pipeline reuses the same scene images captured for calibration —
one physical setup, multiple payoffs.

---

## 5.2 Bundle Adjustment

**The** core optimisation of visual 3D reconstruction. Given observed
projections $\{z_{jk}\}$ of landmarks $\{X_k\}$ in camera frames
$\{T_j\}$, minimise the total reprojection error:

$$
\min_{\{T_j\},\,\{X_k\}} \sum_{(j,k) \in \mathcal{O}}
\left\| z_{jk} - \pi(T_j\,X_k; K_j) \right\|_{\Sigma_{jk}^{-1}}^2
\tag{5.1}
$$

**Reprojection Jacobians.** From $(2.3)$, the projection Jacobian wrt
a 3D point in camera frame is $\partial\pi/\partial\mathbf{X}_c$.
Since $\mathbf{X}_c = R\,\mathbf{X} + t$:

$$
\frac{\partial\pi}{\partial \mathbf{X}} =
\frac{\partial\pi}{\partial\mathbf{X}_c}\,R
\tag{5.2}
$$

For pose perturbation $\xi = (\rho, \phi) \in \mathfrak{se}(3)$, using
the left perturbation $T \mapsto \exp(\hat\xi)\,T$, the induced change
in camera-frame coordinates is
$\delta \mathbf{X}_c \approx \rho - [\mathbf{X}_c]_\times \phi$, hence

$$
\frac{\partial\pi}{\partial\xi} =
\frac{\partial\pi}{\partial\mathbf{X}_c}
\begin{pmatrix} I & -[\mathbf{X}_c]_\times \end{pmatrix}
\tag{5.3}
$$

**Gauss–Newton and normal equations.** Linearising each residual as

$$
r_{jk}(\delta\xi_j, \delta X_k)
\approx r_{jk}^0 + J^{(T)}_{jk}\,\delta\xi_j + J^{(X)}_{jk}\,\delta X_k
\tag{5.4}
$$

and stacking all residuals yields the normal equations
$H\,\delta\theta = -g$ with block-sparse Hessian

$$
H = \begin{pmatrix} H_{TT} & H_{TX} \\ H_{TX}^\top & H_{XX} \end{pmatrix}
\tag{5.5}
$$

**Schur complement** — the key to efficient BA. Since each landmark is
typically seen by only a few cameras, $H_{XX}$ is **block-diagonal**
($3 \times 3$ blocks). Eliminating landmarks:

$$
\underbrace{\left(H_{TT} - H_{TX}\,H_{XX}^{-1}\,H_{TX}^\top\right)}_{\text{reduced camera system}}\,\delta T
= -\left(g_T - H_{TX}\,H_{XX}^{-1}\,g_X\right)
\tag{5.6}
$$

The reduced system has size $6J \times 6J$ (for $J$ cameras) and is
still sparse: only cameras that share landmarks are coupled.

**Gauge freedom.** Monocular BA has a **7-DOF ambiguity**: global
rotation (3), translation (3), and scale (1). Must fix a gauge or add a
regulariser. In the synthetic demo below we hold the first camera fixed
so the remaining problem is well-posed. Stereo or metric depth removes
scale ambiguity; IMU removes roll/pitch.

---

## 5.3 Connection to SLAM

- BA **is** the batch SLAM backend.
- Real-time systems use **sliding-window BA**: marginalise old
  poses/landmarks to bound computation.
- This is exactly the **camera factor** in the overall factor graph.

In [ ]:
all_scene = sorted(Path("assets/scene_images").glob("*.png"))
if len(all_scene) >= 2:
    images = [cv2.imread(str(p)) for p in all_scene[:5]]
    K_sfm = K_est if "K_est" in dir() else K_stereo
    try:
        result = two_view_sfm(images[0], images[1], K_sfm)
        plot_sfm_reconstruction(result)
    except ValueError as exc:
        print(f"SfM demo skipped: {exc}")
else:
    print("Need ≥2 scene images in assets/scene_images/ for SfM demo.")

n_cameras, n_landmarks = 6, 40
ba_result = bundle_adjustment_demo(n_cameras, n_landmarks, rng=rng)
plot_ba_sparsity(ba_result)
plot_ba_convergence(ba_result)

### What can we observe so far?

- Multi-view + BA gives a globally consistent reconstruction, but
  monocular SLAM still has **scale ambiguity**.
- The Schur complement exploits the bipartite structure of BA for
  efficient solving — a pattern that recurs in all SLAM backends.
- Adding **metric depth** (stereo, RGB-D, LiDAR) or **IMU** removes
  the remaining gauge freedoms.

<a id="6"></a>
# 6. Depth Sensing — From Passive to Active

All depth modalities grouped together for direct comparison of
capabilities, noise profiles, and failure modes.

---

## 6.1 Stereo Depth (recap)

From $(4.3)$–$(4.4)$: $Z = fB/d$, noise $\sigma_Z \propto Z^2/(fB)$.
Good at close range, degrades quadratically.

## 6.2 Structured Light

A projector emits a **known IR pattern** (dots, stripes, speckle).
A camera offset from the projector observes the deformed pattern;
triangulation gives depth. Examples: Kinect v1, iPhone TrueDepth
(front camera).

**Failure modes:** sunlight washes out the pattern; absorptive
(black) and transparent (glass) surfaces return no signal.

## 6.3 Time-of-Flight (ToF)

The sensor emits modulated IR light and measures the **phase shift**
$\phi$ of the returning signal:

$$
d = \frac{c}{4\pi f_m}\,\phi
\tag{6.1}
$$

**Unambiguous range:** $d_{\max} = c / (2 f_m)$. Beyond this, phase
wraps around (can be resolved with dual-frequency modulation).

**Noise model:** in the simplified model used here, the **depth-noise
standard deviation** decreases as returned amplitude increases (more
photons → less noise). Systematic errors from **multipath** (light
bouncing off multiple surfaces before returning) bias the measurement
around corners and near reflective objects.

**Failure modes:** multipath near concavities, reflective surfaces
(specular IR reflection), phase wrapping at long range.

## 6.4 LiDAR as Depth Sensor

LiDAR measures time-of-flight with a **narrow laser beam**, mechanically
or electronically scanned across the scene. Much longer range (tens to
hundreds of metres), sparser than camera-based depth, different noise
profile. Detailed treatment in Part 7.

## 6.5 Depth Uncertainty Comparison

| Modality | $\sigma_Z$ growth | Typical range | Key limitation |
|----------|-------------------|---------------|----------------|
| Stereo | $\propto Z^2$ | 0.5–20 m | poor at distance |
| Structured light | $\propto Z^2$ (short range) | 0.2–5 m | sunlight, max range |
| ToF | $\approx$ const (within range) | 0.1–10 m | multipath, phase wrap |
| LiDAR | low, $\approx$ const | 1–200 m | angular resolution, cost |

## 6.6 RGB-D Processing and TSDF Fusion

**Back-projection.** Given pixel $(u, v)$ and depth $Z$, recover 3D:

$$
\mathbf{X} = Z\,K^{-1}\,\tilde{\mathbf{u}}
\tag{6.2}
$$

This produces a **point cloud** per frame. Multiple frames are fused
into a consistent model via **TSDF (Truncated Signed Distance
Function)**:

$$
\phi' = \frac{w\,\phi + w_z\,\phi_z}{w + w_z},
\qquad
w' = \min(w + w_z,\; w_{\max})
\tag{6.3}
$$

where $\phi_z$ is the signed distance from the surface measured by the
new frame, and $w_z$ is its weight (inverse variance). The demo below
isolates this **weighted TSDF update rule** on a synthetic cross-section;
it is analogous in spirit to occupancy-grid evidence accumulation, but
it is not the same probabilistic object as log-odds occupancy.

In [ ]:
plot_depth_comparison(f=500, baselines=[0.06, 0.12], sigma_d=0.5,
                     tof_sigma=0.01, tof_max=10.0,
                     lidar_sigma=0.02, lidar_max=100.0)

tsdf = TSDFFusion(grid_size=(100, 100, 100), voxel_size=0.02, trunc_dist=0.06)
plot_tsdf_evolution(tsdf, n_frames=10, rng=rng)

### What can we observe so far?

- Active depth sensors (structured light, ToF, LiDAR) provide
  **metric depth** directly — removing scale ambiguity.
- In the factor graph, depth adds **unary depth factors** on landmarks
  or direct **point-to-model factors** between scans and the map.
- Each modality has a characteristic noise profile and failure mode;
  the comparison chart guides sensor selection for a given application.

<a id="7"></a>
# 7. LiDAR — Probabilistic Models and Map Building

---

## 7.1 LiDAR Physics

A LiDAR emits short laser pulses and measures the round-trip time
$\tau$; range is $d = c\,\tau / 2$. Scanning mechanisms (rotating
mirror, MEMS mirror, solid-state flash) produce 2D or 3D point clouds.

**Noise sources.**
- *Speckle*: interference of coherent backscattered light.
- *Beam divergence*: footprint grows with range → mixed returns from
  surface edges.
- *Incidence angle*: grazing angles reduce returned intensity and
  increase range noise.
- *Multipath*: glass, water, shiny metal cause spurious short or
  phantom returns.
- *Motion distortion*: platform moves during the scan → geometric
  distortion of the point cloud.

**Failure modes.**
- Glass: beam passes through → phantom points behind.
- Rain/fog: volumetric scattering → short returns, reduced max range.
- Black surfaces: low reflectance → signal below detection threshold.
- Dynamic objects: map corruption from moving obstacles.

---

## 7.2 Beam Model (Thrun)

Given pose $\mathbf{x}$, map $m$, and ray direction, compute the
**expected range** $z^* = h(\mathbf{x}, m)$ to the nearest obstacle.
The measured range $z$ is modelled as a **mixture** of four
physically-motivated components:

**Hit** — sensor noise around the true range (truncated Gaussian):

$$
p_{\mathrm{hit}}(z) = \eta\,\mathcal{N}(z;\, z^*,\, \sigma^2),
\qquad z \in [0,\, z_{\max}]
\tag{7.1a}
$$

**Short** — unexpected close obstacle (exponential):

$$
p_{\mathrm{short}}(z) = \eta'\,\lambda\,e^{-\lambda z},
\qquad z \in [0,\, z^*]
\tag{7.1b}
$$

**Max** — beam returns nothing (point mass at $z_{\max}$):

$$
p_{\mathrm{max}}(z) = \mathbf{1}[z = z_{\max}]
\tag{7.1c}
$$

**Random** — phantom measurement (uniform):

$$
p_{\mathrm{rand}}(z) = 1 / z_{\max},
\qquad z \in [0,\, z_{\max}]
\tag{7.1d}
$$

**Full model:**

$$
p(z \mid \mathbf{x}, m) =
\alpha_{\mathrm{hit}}\,p_{\mathrm{hit}} +
\alpha_{\mathrm{short}}\,p_{\mathrm{short}} +
\alpha_{\mathrm{max}}\,p_{\mathrm{max}} +
\alpha_{\mathrm{rand}}\,p_{\mathrm{rand}}
\tag{7.2}
$$

with $\sum \alpha = 1$. The weights are fit from data via EM.
Typical indoor values: $\alpha_{\mathrm{hit}} \approx 0.9$,
$\alpha_{\mathrm{short}} \approx 0.01$,
$\alpha_{\mathrm{max}} \approx 0.05$,
$\alpha_{\mathrm{rand}} \approx 0.04$.

---

## 7.3 Likelihood Field Model

Instead of ray-tracing for each particle/hypothesis, **precompute** a
**distance transform** $d(\mathbf{p})$ giving the distance from any
point $\mathbf{p}$ to the nearest occupied cell in the map.

For each beam endpoint $\mathbf{p}_i$:

$$
p(z_i \mid \mathbf{x}, m) \propto
\exp\!\left(-\frac{d(\mathbf{p}_i)^2}{2\sigma^2}\right)
\tag{7.3}
$$

**Trade-off:** no explicit modeling of short/max returns, but much
faster to evaluate (no ray tracing) and produces smooth gradients for
optimisation-based approaches.

In [ ]:
beam = BeamModel(z_max=10.0, sigma_hit=0.2, lambda_short=2.0,
                 alpha_hit=0.85, alpha_short=0.05,
                 alpha_max=0.05, alpha_rand=0.05)
plot_beam_model_interactive(beam, z_true=5.0)

## 7.4 Occupancy Grid Mapping

Given known poses $\{\mathbf{x}_t\}$ and LiDAR scans $\{z_t\}$,
estimate the map $m$ as a grid of binary occupancy variables
$m_i \in \{\text{free}, \text{occupied}\}$.

**Binary Bayes filter per cell.** Assuming cells are independent:

$$
p(m_i \mid z_{1:t}, \mathbf{x}_{1:t})
= \frac{
  p(z_t \mid m_i, \mathbf{x}_t)\,p(m_i \mid z_{1:t-1}, \mathbf{x}_{1:t-1})
}{
  p(z_t \mid z_{1:t-1}, \mathbf{x}_{1:t})
}
$$

Define the **odds**
$O_t(m_i) = \frac{p(m_i=\mathrm{occ} \mid z_{1:t}, \mathbf{x}_{1:t})}{p(m_i=\mathrm{free} \mid z_{1:t}, \mathbf{x}_{1:t})}$.
Then Bayes gives

$$
O_t(m_i) = O_{t-1}(m_i)
\frac{p(m_i \mid z_t, \mathbf{x}_t)}{1 - p(m_i \mid z_t, \mathbf{x}_t)}
\frac{1 - p_{\mathrm{prior}}}{p_{\mathrm{prior}}}
$$

Taking logs yields the additive **log-odds** update. Define
$l(m_i) = \log \frac{p(m_i = \text{occ})}{p(m_i = \text{free})}$. Then

$$
l_t(m_i) = l_{t-1}(m_i)
+ \underbrace{\log \frac{p(m_i \mid z_t, \mathbf{x}_t)}{1 - p(m_i \mid z_t, \mathbf{x}_t)}}_{\text{inverse sensor model}}
- l_0
\tag{7.4}
$$

where $l_0 = \log\frac{p_{\text{prior}}}{1 - p_{\text{prior}}}$ is the
prior log-odds (typically $l_0 = 0$ for $p_{\text{prior}} = 0.5$).

**Inverse sensor model.** For each beam:
- Cells along the ray **before** the endpoint: increase $p(\text{free})$.
- Cell **at** the endpoint: increase $p(\text{occ})$.
- Cells **beyond** the endpoint: no update.

**Ray tracing** (Bresenham's line algorithm) efficiently identifies
which cells a beam passes through.

---

## 7.5 Point Cloud Registration: ICP

**Iterative Closest Point** aligns two point clouds by alternating
between (a) finding closest-point correspondences and (b) solving for
the rigid transform.

**Point-to-point.** Given correspondences
$(p_i, q_{c(i)})$:

$$
\min_{R, t} \sum_{i=1}^{N} \|R\,p_i + t - q_{c(i)}\|^2
\tag{7.5}
$$

After centering the clouds,
$\tilde p_i = p_i - \bar p$ and $\tilde q_i = q_{c(i)} - \bar q$, the
cross-covariance is

$$
W = \sum_i \tilde p_i\,\tilde q_i^\top = U\Sigma V^\top
$$

and the optimal rotation is

$$
R = V\,\mathrm{diag}(1, 1, \det(VU^\top))\,U^\top,
\qquad
t = \bar q - R\,\bar p
\tag{7.6}
$$

The determinant correction prevents an improper reflection when the SVD
would otherwise return $\det(R) < 0$.

**Point-to-plane** variant — project the error onto the target normal
$n_{c(i)}$:

$$
\min_{R, t} \sum_{i=1}^{N}
\bigl((R\,p_i + t - q_{c(i)}) \cdot n_{c(i)}\bigr)^2
\tag{7.7}
$$

Converges faster (fewer iterations) but requires surface normals.

**Failure modes.** ICP finds **local** minima — converges to wrong
alignment if initial guess is poor. Symmetric environments (long
corridors, round rooms) give degenerate cost landscapes. Sparse scans
have too few constraints.

In [ ]:
# ── Live Demo: iPhone LiDAR Point Cloud ─────────────────────────────────────
# Receives depth frames + camera poses from PerceptionDemo app.
# Backprojects to a 3D point cloud and visualises in Open3D (if available)
# or matplotlib. Falls back to pre-recorded PLY if no iPhone is connected.
#
# Start PerceptionDemo on iPhone with Depth stream enabled.
# Stop with the ■ (interrupt kernel) button.
# ─────────────────────────────────────────────────────────────────────────────
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from lib.phone_stream import PhoneUSBSource, load_depth_ply
from lib.depth import backproject_depth_frame
from lib.viz import create_live_depth_plot

_depth_phone = PhoneUSBSource()
_depth_phone.start()
print("Waiting up to 15 s for iPhone USB connection…")
connected = _depth_phone.wait_for_connection(timeout=15.0)

_depth_plot = create_live_depth_plot()
_depth_plot.show()

accumulated_pts: list[np.ndarray] = []

if connected:
    print("iPhone connected — streaming depth frames.")
    try:
        for _ in range(200):  # ~100 s at 0.5 s polling
            pkt = _depth_phone.get_latest_depth()
            if pkt is not None:
                _depth_plot.update(pkt)
                pts = backproject_depth_frame(
                    pkt.depth_m, pkt.intrinsics,
                    T_cw=pkt.pose_T_cw, max_depth=5.0, subsample=8,
                )
                accumulated_pts.append(pts)
            time.sleep(0.5)
    finally:
        _depth_phone.stop()
        _depth_plot.close()
else:
    print("No iPhone — checking for pre-recorded PLY…")
    _depth_phone.stop()
    ply_pts = load_depth_ply()
    if ply_pts is not None:
        accumulated_pts = [ply_pts]
        print(f"Loaded {len(ply_pts)} points from PLY.")
    else:
        print("No replay data available. Skipping 3D visualisation.")

# 3D scatter of accumulated point cloud
if accumulated_pts:
    all_pts = np.concatenate(accumulated_pts, axis=0)
    if len(all_pts) > 5000:
        idx = np.random.choice(len(all_pts), 5000, replace=False)
        all_pts = all_pts[idx]
    try:
        import open3d as o3d
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(all_pts)
        o3d.visualization.draw_geometries([pcd], window_name="iPhone LiDAR", width=800, height=600)
    except ImportError:
        fig3d = plt.figure(figsize=(8, 6))
        ax3d = fig3d.add_subplot(111, projection="3d")
        ax3d.scatter(all_pts[:, 0], all_pts[:, 1], all_pts[:, 2],
                     s=0.5, c=all_pts[:, 2], cmap="plasma", alpha=0.6)
        ax3d.set_title(f"iPhone LiDAR — {len(all_pts)} points")
        ax3d.set_xlabel("X (m)"); ax3d.set_ylabel("Y (m)"); ax3d.set_zlabel("Z (m)")
        plt.tight_layout(); plt.show()

In [ ]:
scan_files = sorted(Path("assets/lidar_scans").glob("*.npz"))
if scan_files:
    scans = [np.load(str(f)) for f in scan_files]
else:
    scans = OccupancyGrid.generate_synthetic_scans(n_scans=20, rng=rng)

grid = OccupancyGrid(size=(200, 200), resolution=0.05)
plot_occupancy_grid_evolution(grid, scans)

source, target = OccupancyGrid.synthetic_icp_clouds(rng=rng)
R_icp, t_icp, errors = icp_point_to_point(source, target, max_iter=30)
plot_icp_convergence(source, target, R_icp, t_icp, errors)

### What can we observe so far?

- LiDAR provides **dense metric geometry** at long range.
- The beam model gives a principled likelihood $p(z \mid \mathbf{x}, m)$
  for localisation; the inverse model builds occupancy grids for
  planning.
- ICP/NDT add **scan-matching factors** between consecutive poses.
- This completes the sensor catalogue. Part 8 assembles everything.

<a id="8"></a>
# 8. SLAM Systems and Sensor Fusion

---

## 8.1 The Residual Zoo — Same Framework, Different Sensors

Every sensor we studied contributes a **residual** to the same
optimisation problem. The table below is the payoff of the unified
template:

| Sensor | Residual $r$ | Noise $\Sigma$ | Factor connects |
|--------|-------------|----------------|-----------------|
| Camera reprojection | $z_{jk} - \pi(T_j X_k)$ | $\sim 1$–$2\;\text{px}$ | pose ↔ landmark |
| Stereo disparity | $d_{\text{obs}} - fB/Z$ | constant in $d$ | pose ↔ landmark depth |
| LiDAR beam | $z - h(\mathbf{x}, m)$ | mixture model | pose ↔ map |
| ICP point-to-plane | $(R p_i + t - q) \cdot n$ | range + angular | pose ↔ pose |
| IMU preintegration | $(\Delta\hat R, \Delta\hat v, \Delta\hat p) \ominus \text{predicted}$ | integration noise | pose ↔ pose + bias |
| Encoder odometry | $\mathrm{Log}(\hat T_{i,i+1}^{-1} T_i^{-1} T_{i+1})$ | $\propto$ path length | pose ↔ pose |
| GPS | $z_{\text{GPS}} - p$ | $\sim 1$–$10\;\text{m}$ | unary on pose |

All enter the same MAP estimation:

$$
\hat X = \arg\min_X \sum_j \|r_j(X)\|_{\Sigma_j^{-1}}^2
\tag{8.1}
$$

---

## 8.2 Factor Graph Formulation

**Nodes:** poses $\{T_i\}$, velocities, IMU biases, landmarks $\{X_k\}$,
(optionally) calibration parameters.

**Factors:** one per residual above. Each factor's Jacobian and noise
model come directly from the sensor derivations in Parts 1–7.

**Solving:**
- *Batch:* full Gauss–Newton / LM over all variables (offline).
- *Incremental:* iSAM2 (Bayes tree), Ceres with sliding window.
- *Marginalization:* remove old variables while preserving their
  information as a prior on the remaining ones. Keeps the problem
  size bounded for real-time operation.

---

## 8.3 Visual SLAM Pipelines

**ORB-SLAM3** (feature-based):
1. *Tracking:* match ORB features frame-to-frame, motion-only BA.
2. *Local mapping:* triangulate new points, local BA.
3. *Loop closure:* bag-of-words place recognition → global BA.
4. Supports mono / stereo / RGB-D / inertial modes.

**DSO** (direct, semi-dense):
- Minimises **photometric error** $\sum |I_1(\mathbf{p}) - I_2(\text{warp}(\mathbf{p}))|^2$.
- Jointly optimises inverse-depth and pose.
- No explicit features; works with any image gradient.

**Trade-offs:** ORB-SLAM3 robust to illumination changes but needs
texture; DSO works with weaker texture but is sensitive to photometric
violations (auto-exposure, rolling shutter).

---

## 8.4 LiDAR SLAM Landscape

- **2D:** GMapping (RBPF + occupancy grid), Hector SLAM (scan-matching,
  no odometry), Cartographer (submaps + pose graph + branch-and-bound
  loop closure).
- **3D:** LOAM / LeGO-LOAM (edge + planar features for fast odometry),
  NDT-based methods.
- **NDT matching:** approximate local point distribution as
  $\mathcal{N}(\mu_j, \Sigma_j)$ per cell; scan likelihood is
  $\prod_i \mathcal{N}(p_i \mid \mu_j, \Sigma_j)$. Smooth cost
  landscape, fast convergence.

---

## 8.5 Visual-Inertial Odometry (VIO)

**Factor graph** with:
- Camera reprojection factors (Part 5).
- IMU preintegration factors: integrate $\Delta R, \Delta v, \Delta p$
  between keyframes; store Jacobians wrt bias for cheap
  re-linearisation when bias estimate changes.

**Why VIO works:**
- IMU constrains **scale** (via acceleration) and **roll/pitch** (via
  gravity). Camera constrains **lateral drift** and **yaw**.
- Together they resolve the monocular scale ambiguity and provide
  metric, drift-bounded odometry.

**Observability.** VIO recovers metric scale and gravity direction that
monocular SLAM alone cannot — but only when there is sufficient
rotational/translational excitation. Constant-velocity straight-line
motion is degenerate.

---

## 8.6 Failure Mode Gallery

| Scenario | Primary failure | Sensor affected | Mitigation |
|----------|----------------|-----------------|------------|
| Low / repetitive texture | feature matching fails | camera | add LiDAR or direct method |
| Textureless corridor | stereo disparity holes | stereo | add LiDAR or IMU dead-reckoning |
| Glass / mirrors | pass-through / multipath | LiDAR | camera (visual features on glass surface) |
| Fast rotation | motion blur, IMU saturation | camera + IMU | higher frame rate, global shutter |
| Forward-only motion | degenerate epipolar geometry | monocular camera | stereo or IMU (scale from accel) |
| Long featureless tunnel | no loop closure, drift | all visual | LiDAR + IMU, or GPS if available |
| Dynamic objects | map corruption, wrong associations | all | semantic filtering, robust kernels |
| Darkness | no image features | camera | LiDAR (active illumination), thermal |

> **Key insight:** no single sensor handles all scenarios. Robust
> systems fuse complementary modalities — the factor graph framework
> makes this natural.

---

## 8.7 Modern Frontiers

- **Neural implicit maps** — NeRF-SLAM, PIN-SLAM: replace explicit
  grids/points with neural radiance fields; compact, differentiable,
  but computationally expensive.
- **Semantic SLAM** — object-level landmarks (chairs, doors) for
  long-term stability and human-understandable maps.
- **Foundation models for perception** — DINOv2 features for
  place recognition, SAM for zero-shot segmentation.
- **Tightly-coupled multi-sensor** — BAMF-SLAM (multi-fisheye + IMU),
  MAVIS (multi-camera VIO on $\mathrm{SE}_2(3)$).

In [ ]:
plot_factor_graph_annotated()
plot_failure_gallery()

## Observability Summary

| After part | What is observable | What is still ambiguous |
|------------|--------------------|------------------------|
| 1 — Proprioception | relative motion; roll/pitch from gravity when static | global position, yaw, drift-free velocity |
| 2 — Camera + ArUco | bearing to features; full 6-DOF if landmarks known | depth for generic monocular landmarks |
| 4 — Two-view | 3D structure up to scale | global scale, 7-DOF gauge |
| 5 — BA | globally consistent reconstruction | scale (mono), global gauge |
| 6 — Active depth | metric depth | pose still needs registration over time |
| 7 — LiDAR | dense metric geometry, occupancy | global consistency still needs scan matching / loop closure |
| 8 — Fusion | stronger local metric pose and map estimates | global pose still needs loop closure or absolute references |

> **Take-away.** Perception is the art of choosing sensors whose
> observability properties **complement** each other, and combining
> their residuals in a unified optimisation framework.